# jax_gpt.py on a free Cloud TPU (Colab or Kaggle)

Runs the same JAX GPT benchmark as on the RTX 5070 (`jax-xla/README.md`).

**Colab:** Runtime > Change runtime type > **v5e-1 TPU** (or v2-8). **Kaggle:** Settings > Accelerator > **TPU v5e-8**.
Then Run all. The last cell prints one JSON line per run: copy those lines back.

In [ ]:
import jax, subprocess, sys
print(jax.__version__, jax.default_backend(), jax.devices())

In [ ]:
!pip -q install optax
!rm -rf gsl && git clone -q --depth 1 -b jax-xla-port https://github.com/zakiscoding/gpu-systems-lab gsl
%cd gsl

In [ ]:
# 1 TPU core, same shapes as the GPU table (bf16, XLA attention: cuDNN is GPU-only)
!python jax-xla/jax_gpt.py --d-model 768 --n-layers 12 --n-heads 12 --seq 1024 --batch 4 --vocab 32768 --steps 60 --warmup 10 --out tpu_1core_b4.json

In [ ]:
# all local TPU cores, data parallel: batch 4 per core
n = jax.device_count()
!python jax-xla/jax_gpt.py --d-model 768 --n-layers 12 --n-heads 12 --seq 1024 --batch {4*n} --vocab 32768 --steps 60 --warmup 10 --data-parallel --out tpu_dp.json

In [ ]:
import json, glob
for f in sorted(glob.glob('tpu_*.json')):
    r = json.load(open(f)); r.pop('losses', None)
    print(json.dumps(r))